# INSTALL DEPS

In [3]:
!pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 34.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=4503265 sha256=80da327248e89d56508a029f004b4c1de29677856dde81ac121719e6c411078a
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [39]:
!pip install --user langchain

In [49]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.2 MB/s eta 0:00:00:00:0100:01


In [ ]:
!pip install pypdf2

In [40]:
import langchain
print(langchain.__version__)

1.2.4


# **METHOD 1**

# LOAD local LLM

### here we use teh downloaded DeepSeek-R1-Distill-Llama-8B

In [152]:
from llama_cpp import Llama

model_path = "/kaggle/input/models/alaeeladlani/deepseek/gguf/default/1/DeepSeek-R1-Distill-Llama-8B-Q4_0.gguf"
llm = Llama(model_path=model_path)

llama_model_loader: loaded meta data with 32 key-value pairs and 292 tensors from /kaggle/input/models/alaeeladlani/deepseek/gguf/default/1/DeepSeek-R1-Distill-Llama-8B-Q4_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = DeepSeek R1 Distill Llama 8B
llama_model_loader: - kv   3:                           general.basename str              = DeepSeek-R1-Distill-Llama
llama_model_loader: - kv   4:                         general.size_label str              = 8B
llama_model_loader: - kv   5:                          llama.block_count u32              = 32
llama_model_loader: - kv   6:                       llama.context_lengt

# LOAD PDF OR context DATA

### use a pdf or any type of text data

In [153]:
from PyPDF2 import PdfReader

def load_pdf_text(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# DEVIDE data into Chunks

In [154]:
import re

def enhanced_text_chunker(text, chunk_size=30, overlap=5):
    # Split text into sentences while keeping separators
    sentences = re.split(r'(?<=[.!?\n])\s+', text)
    chunks = []
    current_chunk = []
    current_len = 0
    for sentence in sentences:
        words = sentence.split()
        if current_len + len(words) > chunk_size:
            # finish current chunk
            if current_chunk:
                chunks.append(" ".join(current_chunk).strip())
            # start new chunk with overlap words
            if overlap > 0 and current_chunk:
                overlap_words = sum([s.split() for s in current_chunk], [])
                current_chunk = [" ".join(overlap_words[-overlap:])] if overlap_words else []
                current_len = sum(len(s.split()) for s in current_chunk)
            else:
                current_chunk = []
                current_len = 0
        current_chunk.append(sentence)
        current_len += len(words)
    
    if current_chunk:
        chunks.append(" ".join(current_chunk).strip())    
    return chunks

### we use this pdf here https://content.vu.edu.au/sites/default/files/sample-research-report.pdf

In [155]:
path = "/kaggle/input/datasets/alaeeladlani/rag-data/sample-research-report.pdf"
document_text = load_pdf_text(path) 
chunks = enhanced_text_chunker(document_text, chunk_size=100, overlap=10)
print("Number of chunks:", len(chunks))

Number of chunks: 42


In [156]:
chunks[1]

'Australian workforce since the end of World War II (1945-2000). A review  of some of th\ne available literature \nprovides insights into the changi ng role of women and migrants in the workforce, and the \ninfluence of new technologies and changing levels of unemployment have also been \nconsidered. Key findings include:  \nThere has been a marked \nincrease in wo men’s participation in the workforce, \nparticularly that of married women.'

### SentenceTransformer → turns text into vectors (numbers)

### FAISS → super-fast similarity search on vectors

### NumPy → numeric arrays (FAISS needs this)

In [157]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

#Loads a pretrained sentence embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks, convert_to_numpy=True)
dimension = embeddings.shape[1]
#Create a vector database using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### for a given question, select the top-k closest vectors

In [160]:
question = "What government or institution is mentioned as a data source?"

def retrieve_chunks(query, top_k=10):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_vec, top_k)
    return [chunks[i] for i in I[0]]
    
context_chunks = retrieve_chunks(question, top_k=10)
len(context_chunks)

10

## DEFINE THE CONTEXT

In [163]:
def trim_context(text, max_chars=1200):
    return text[:max_chars]
    
def ask_rag_single(question, top_k=10):
    context_chunks = retrieve_chunks(question, top_k=top_k)
    context_text = "\n".join(context_chunks)
    context_text = trim_context(context_text, max_chars=1200)
    prompt = f"""
You are an information extraction system.

Rules:
- Answer ONLY using the context
- Output ONE line
- NO explanations
- NO extra text

Context:
{context_text}

Question:
{question}

Answer:
""".strip()

    resp = llm(prompt,max_tokens=64,temperature=0.0,stop=["\n"])
    return resp["choices"][0]["text"].strip()

In [164]:
question = "What government or institution is mentioned as a data source?"
answer = ask_rag_single(question)
print(answer)

llama_perf_context_print:        load time =   31897.97 ms
llama_perf_context_print: prompt eval time =   31897.29 ms /   320 tokens (   99.68 ms per token,    10.03 tokens per second)
llama_perf_context_print:        eval time =    3535.12 ms /    11 runs   (  321.37 ms per token,     3.11 tokens per second)
llama_perf_context_print:       total time =   35448.19 ms /   331 tokens
llama_perf_context_print:    graphs reused =         10


The Australian Bureau of Statistics is mentioned as a data source.


----
-
---
-
----
-
----

# **METHOD 2**

# install deps

In [81]:
!pip install -U transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00:00:0100:01


In [85]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

### use the Qwen/Qwen2.5-3B-Instruct llm

In [86]:
model_id = "Qwen/Qwen2.5-3B-Instruct"  # example

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.float16)
model.eval()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

### SIMPLE USE CASE

In [91]:
# define simple prompt
prompt = """Answer in ONE line only.
            Context:
                    Australian Bureau of Statistics provides workforce data.
            Question:
                    What institution is the data source?
            Answer:"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=32, do_sample=False )
print(tokenizer.decode(out[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Answer in ONE line only.
            Context:
                    Australian Bureau of Statistics provides workforce data.
            Question:
                    What institution is the data source?
            Answer: Australian Bureau of Statistics is the data source.


In [90]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs,max_new_tokens=64,do_sample=True,temperature=0.7,top_p=0.9)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Answer in ONE line only.
            Context:
                    Australian Bureau of Statistics provides workforce data.
            Question:
                    What institution is the data source?
            Answer: The Australian Bureau of Statistics is the data source.


# USE LONG CONTEXT

In [170]:
from PyPDF2 import PdfReader

data_path = "/kaggle/input/datasets/alaeeladlani/rag-data/sample-research-report.pdf"
#Load PDF
reader = PdfReader(data_path)

#find the number of pages
num_pages = len(reader.pages)
print("Number of pages in PDF:", num_pages)

#preview text of first page
first_page_text = reader.pages[0].extract_text()
print("First page text preview:\n", first_page_text[:50])

Number of pages in PDF: 12
First page text preview:
 Unit 4: Report Writing 
Research Report 
 
 
  
 



In [172]:
#load the data
data_path = "/kaggle/input/datasets/alaeeladlani/rag-data/sample-research-report.pdf"
document_text = load_pdf_text(data_path)

In [173]:
#divide into chunks 
chunks = enhanced_text_chunker(document_text, chunk_size=100, overlap=0)
print("Number of chunks:", len(chunks))

Number of chunks: 36


In [174]:
#embed the chunks
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(chunks, convert_to_numpy=True)
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [175]:
#find closer chunks to the query
def retrieve_chunks(query, top_k=10):
    query_vec = embed_model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_vec, top_k)
    return [chunks[i] for i in I[0]]

question = "What government or institution is mentioned as a data source?"
context_chunks = retrieve_chunks(question, top_k=10)
len(context_chunks)

10

In [176]:
#print the closest chunks
for i, c in enumerate(context_chunks):
    print("__________________________________________________")
    print(f"Chunk {i+1}: {c}")
    print("--------------------------------------------------")

__________________________________________________
Chunk 1: The 
reliance on secondary sources has resulted in some patchy data. For example, it is not 
possible to identify for any given year a br eakdown of the Australian workforce by the 
following categories:  
• unmarried Australia-born women 
• married Australia-born women 
• unmarried Australia-born men 
• married Australia-born men 
• unmarried immigrant women 
• married immigrant women 
• unmarried immigrant men 
• married immigrant men 
 Greater access to primary data would enable a mo
re thorough analysis to be made. Version 1.0 
Concurrent Study Research Report Page 10 
 5. Reference List
--------------------------------------------------
__________________________________________________
Chunk 2: Where the infor mation refers to a particular state, this  will be noted. The 
period under consideration is 1945 to 2000, although where available data does not cover the entire period, this is stated. The re port focuses on seve

## RAG PROMPT

In [138]:
def build_rag_prompt(context_chunks, question):
    context = "\n\n".join(context_chunks)
    return f"""
You are an information extraction system.

Rules:
- Use ONLY the context below.
- If the answer is not explicitly stated, output: NOT FOUND
- Output ONE short sentence only.
- Do NOT explain.
- Do NOT add extra text.

Context:
{context}

Question:
{question}

Answer:
"""

In [139]:
question = "What government or institution is mentioned as a data source?"
context_chunks = retrieve_chunks(question, top_k=10)

prompt = build_rag_prompt(context_chunks, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs,max_new_tokens=40,do_sample=False,repetition_penalty=1.1)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only what comes after "Answer:"
final_answer = answer.split("Answer:")[-1].strip()
print(final_answer)

The Australian Bureau of Statistics is mentioned as a data source.
NOT_FOUND

The provided context does not mention any specific government or institution other than the Australian Bureau of Statistics as a data source. The Australian


In [141]:
answer = final_answer.splitlines()[0].strip()
print(answer)

The Australian Bureau of Statistics is mentioned as a data source.


## RAG PROMPT 2

In [142]:
def build_rag_prompt(context_chunks, question):
    context = "\n\n".join(context_chunks)
    return f"""
You are an information extraction system.

Rules:
- Use ONLY the context.
- Output ONLY the answer.
- ONE line only.
- No punctuation.
- No explanations.
- If not explicitly stated, output exactly: NOT_FOUND

Context:
{context}

Question:
{question}

Answer:
"""

In [146]:
question = "What government or institution is mentioned as a data source?"
context_chunks = retrieve_chunks(question, top_k=10)
prompt = build_rag_prompt(context_chunks, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50,do_sample=False,repetition_penalty=1.,eos_token_id=tokenizer.eos_token_id,)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = decoded.split("Answer:")[-1].strip()
print(answer)

Australian Bureau of Statistics
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND
NOT_FOUND


In [147]:
answer = answer.splitlines()[0].strip()
print(answer)

Australian Bureau of Statistics


# NEW QUESTION

In [150]:
question = "What did or can cause unemployment according to context?"
context_chunks = retrieve_chunks(question, top_k=10)
prompt = build_rag_prompt(context_chunks, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs,max_new_tokens=40,do_sample=False,repetition_penalty=1.1)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Extract only what comes after "Answer:"
final_answer = answer.split("Answer:")[-1].strip()
print(final_answer)

age, proficiency in speaking English, geographic location, gender, lack of proficiency in English, undervaluing or lack of recognition of qualifications received overseas, lack of a verifiable employment 'history', discrimination


In [148]:
question = "What did or can cause unemployment according to context?"
context_chunks = retrieve_chunks(question, top_k=10)
prompt = build_rag_prompt(context_chunks, question)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50,do_sample=False,repetition_penalty=1.,eos_token_id=tokenizer.eos_token_id,)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer = decoded.split("Answer:")[-1].strip()
print(answer)

age, proficiency in speaking English, geographic location, lack of proficiency in English, undervaluing or lack of recognition of qualifications received overseas, lack of a verifiable employment history, discrimination, under-representation in trade unions, lack of proficiency in English
